# SPilot VL Colab Server

Colab Pro에서 Qwen VL 모델을 OpenAI-compatible FastAPI 서버로 띄우는 노트북입니다.

- 추천 모델: `Qwen/Qwen2.5-VL-32B-Instruct`
- VRAM 부족 시 fallback: `Qwen/Qwen2.5-VL-7B-Instruct`
- 로컬 백엔드는 `/v1/chat/completions` 엔드포인트로 contact sheet 이미지를 보냅니다.


In [ ]:
!pip install -q git+https://github.com/huggingface/transformers accelerate fastapi uvicorn pyngrok nest_asyncio pillow qwen-vl-utils[decord]==0.0.8


In [ ]:
import os, json, base64, re, threading, nest_asyncio
from io import BytesIO
from PIL import Image
import torch
from fastapi import FastAPI, Request
from fastapi.middleware.cors import CORSMiddleware
from pyngrok import ngrok
import uvicorn
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

nest_asyncio.apply()


In [ ]:
# Colab Pro 권장: 32B. VRAM 부족 시 7B로 변경하세요.
MODEL_ID = os.environ.get("MODEL_ID", "Qwen/Qwen2.5-VL-32B-Instruct")
# MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=256*28*28, max_pixels=1280*28*28)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)
print("loaded", MODEL_ID)


In [ ]:
SYSTEM_HINT = """당신은 건설현장 CCTV 사고 분석 VL 모델입니다. 반드시 JSON만 출력하세요.
필수 필드: primary_type, secondary_type, injured_count, cause, confidence, timeline, workers, evidence, details.
사고 유형은 추락, 낙상, 화재, 기타 중 하나입니다. 부상자 수는 영상에서 사고와 직접 관련된 피해자 후보만 보수적으로 세세요.
원인은 영상에서 관찰 가능한 시간순 흐름만 쓰고 법적 책임, 승인 여부, 사망 여부는 단정하지 마세요.
"""

def normalize_messages(messages):
    normalized = []
    for message in messages:
        content = []
        for item in message.get("content", []):
            if item.get("type") == "text":
                content.append({"type": "text", "text": SYSTEM_HINT + "\n\n" + item.get("text", "")})
            elif item.get("type") == "image_url":
                url = item.get("image_url", {}).get("url", "")
                content.append({"type": "image", "image": url})
            elif item.get("type") == "video":
                content.append(item)
        normalized.append({"role": message.get("role", "user"), "content": content})
    return normalized

def extract_json(text):
    match = re.search(r"\{.*\}", text, re.S)
    return match.group(0) if match else text


In [ ]:
app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

@app.get("/health")
def health():
    return {"status": "ok", "model": MODEL_ID}

@app.post("/v1/chat/completions")
async def chat_completions(request: Request):
    body = await request.json()
    messages = normalize_messages(body.get("messages", []))
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs, video_kwargs = process_vision_info(messages, return_video_kwargs=True)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt", **video_kwargs).to(model.device)
    with torch.inference_mode():
        generated_ids = model.generate(**inputs, max_new_tokens=int(body.get("max_tokens", 700)), do_sample=False)
    trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
    output_text = processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
    output_text = extract_json(output_text)
    return {"id": "spilot-vl", "object": "chat.completion", "model": MODEL_ID, "choices": [{"index": 0, "message": {"role": "assistant", "content": output_text}, "finish_reason": "stop"}]}


In [ ]:
# 선택: ngrok authtoken이 있으면 아래 줄 주석 해제
# ngrok.set_auth_token("YOUR_NGROK_TOKEN")
public_url = ngrok.connect(8000, "http")
print("LLM_API_BASE=" + str(public_url) + "/v1")

thread = threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000), daemon=True)
thread.start()
